# Day 3 — Inspect decision datasets

Goal: load `avbiswas/bev-decision` and `tau/commonsense_qa`, print columns + a few rows, and understand the raw schema before we normalize into `(state, options, correct_idx)`.

In [1]:
from datasets import load_dataset
from pprint import pprint

## 1. `avbiswas/bev-decision`

Main training / eval source for choice-style questions.

In [2]:
bev = load_dataset("avbiswas/bev-decision")
print("splits:", list(bev.keys()))
for split_name, split in bev.items():
    print(f"  {split_name}: {len(split):,} rows")

README.md:   0%|          | 0.00/18.8k [00:00<?, ?B/s]

data/train.parquet: reconstructing file:   0%|          |  0.00B / 31.7MB            

data/train.parquet: downloading bytes:           |  0.00B            

data/test.parquet: reconstructing file:   0%|          |  0.00B / 5.75MB            

data/test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/125614 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/24386 [00:00<?, ? examples/s]

splits: ['train', 'test']
  train: 125,614 rows
  test: 24,386 rows


In [3]:
bev_split = bev["train"] if "train" in bev else bev[list(bev.keys())[0]]
print("columns:", bev_split.column_names)
print("features:")
pprint(bev_split.features)

columns: ['state', 'questions_json', 'domain', 'question_types', 'question_count']
features:
{'domain': Value('string'),
 'question_count': Value('int16'),
 'question_types': List(Value('string')),
 'questions_json': Value('large_string'),
 'state': Value('large_string')}


In [4]:
print("--- 5 sample rows ---")
for i, row in enumerate(bev_split.select(range(5))):
    print(f"\n=== row {i} ===")
    pprint(dict(row))

--- 5 sample rows ---

=== row 0 ===
{'domain': 'Scientific and paper understanding',
 'question_count': 1,
 'question_types': ['choice'],
 'questions_json': '{"citation_intent":{"type":"choice","instructions":"What '
                   'is the intent of this '
                   'citation?","criteria":{"background":"Background or prior '
                   'work","method":"Method or procedure","result":"Result or '
                   'finding"},"label":"background"}}',
 'state': 'To find novel genes in the FHV-1 genome ab initio, and as a second '
          'approach to verify the annotation, the sequence was submitted to '
          'GeneMarkS (Besemer et al., 2001).'}

=== row 1 ===
{'domain': 'Scientific and paper understanding',
 'question_count': 1,
 'question_types': ['noul'],
 'questions_json': '{"decision":{"type":"noul","instructions":"Is the answer '
                   'to the biomedical question yes, given the supplied '
                   'evidence?","label":true}}',
 'sta

## 2. `tau/commonsense_qa`

Clean 5-option multiple choice — good secondary eval set.

In [5]:
csqa = load_dataset("tau/commonsense_qa")
print("splits:", list(csqa.keys()))
for split_name, split in csqa.items():
    print(f"  {split_name}: {len(split):,} rows")

README.md:   0%|          | 0.00/7.39k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.25MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  160kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  151kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9741 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1221 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1140 [00:00<?, ? examples/s]

splits: ['train', 'validation', 'test']
  train: 9,741 rows
  validation: 1,221 rows
  test: 1,140 rows


In [6]:
csqa_split = csqa["train"] if "train" in csqa else csqa[list(csqa.keys())[0]]
print("columns:", csqa_split.column_names)
print("features:")
pprint(csqa_split.features)

columns: ['id', 'question', 'question_concept', 'choices', 'answerKey']
features:
{'answerKey': Value('string'),
 'choices': {'label': List(Value('string')), 'text': List(Value('string'))},
 'id': Value('string'),
 'question': Value('string'),
 'question_concept': Value('string')}


In [7]:
print("--- 5 sample rows ---")
for i, row in enumerate(csqa_split.select(range(5))):
    print(f"\n=== row {i} ===")
    pprint(dict(row))

--- 5 sample rows ---

=== row 0 ===
{'answerKey': 'A',
 'choices': {'label': ['A', 'B', 'C', 'D', 'E'],
             'text': ['ignore',
                      'enforce',
                      'authoritarian',
                      'yell at',
                      'avoid']},
 'id': '075e483d21c29a511267ef62bedc0461',
 'question': 'The sanctions against the school were a punishing blow, and they '
             'seemed to what the efforts the school had made to change?',
 'question_concept': 'punishing'}

=== row 1 ===
{'answerKey': 'B',
 'choices': {'label': ['A', 'B', 'C', 'D', 'E'],
             'text': ['race track',
                      'populated areas',
                      'the desert',
                      'apartment',
                      'roadblock']},
 'id': '61fe6e879ff18686d7552425a36344c8',
 'question': 'Sammy wanted to go to where the people were.  Where might he go?',
 'question_concept': 'people'}

=== row 2 ===
{'answerKey': 'A',
 'choices': {'label': ['A', 'B', 'C'

## 3. Scratch notes

After looking at both, jot what maps to:

| Our field | bev-decision column | CommonsenseQA column |
|---|---|---|
| `state` | ? | ? |
| `options` | ? | ? |
| `correct_idx` | ? | ? |

Next: write `src/data.py` that does this conversion and saves seeded 200-row eval splits.